# Day 6 · Exercise 3: Few-Shot Example Builder

**What you'll build:** `build_few_shot_prompt(task_description: str, examples: list[dict], new_input: str) -> str` — a function that assembles a formatted few-shot prompt string from a task description, a list of input/output example dicts, and a new input to process.

**Why it matters:** Writing few-shot prompts by hand is fragile and repetitive; a reusable builder lets you swap tasks and examples without ever touching the prompt structure.

## Your Implementation

In [ ]:
def build_few_shot_prompt(task_description: str, examples: list[dict], new_input: str) -> str:
    """Build a few-shot prompt string from a task description, examples, and a new input.

    Each dict in `examples` must have exactly two keys:
        - "input":  the example input text
        - "output": the ideal output for that input

    The returned string has this structure:
        Task: <task_description>

        Example 1:
        Input: <examples[0]["input"]>
        Output: <examples[0]["output"]>

        Example 2:
        Input: <examples[1]["input"]>
        Output: <examples[1]["output"]>

        ... (one block per example)

        Now do the same for:
        Input: <new_input>
        Output:

    Args:
        task_description: One sentence describing the task (e.g. "Classify the sentiment").
        examples: List of dicts, each with keys 'input' and 'output', providing
            worked demonstrations of the task. 2-5 examples recommended.
        new_input: The actual input you want the model to process.

    Returns:
        A formatted prompt string ready to be sent as a user message.

    Example:
        >>> examples = [
        ...     {"input": "I love this!", "output": "POSITIVE"},
        ...     {"input": "Terrible service.", "output": "NEGATIVE"},
        ... ]
        >>> prompt = build_few_shot_prompt("Classify the sentiment.", examples, "It was okay.")
        >>> print(prompt)
        Task: Classify the sentiment.
        ...
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work

Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score, total = 0, 5

    # Check 1: function exists and is callable
    try:
        assert callable(build_few_shot_prompt), 'build_few_shot_prompt is not defined or not callable'
        print(f'{_PASS} Check 1/{total}: function exists and is callable')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 1/{total}: {e}')
        return

    # Check 2: return type is str
    try:
        examples = [
            {"input": "I love this!", "output": "POSITIVE"},
            {"input": "Terrible service.", "output": "NEGATIVE"},
        ]
        result = build_few_shot_prompt("Classify the sentiment.", examples, "It was okay.")
        assert isinstance(result, str), f'expected str, got {type(result).__name__}'
        print(f'{_PASS} Check 2/{total}: return type is str')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 2/{total}: {e}')
        return

    # Check 3: task description appears in the output
    try:
        task = "Classify the sentiment."
        examples = [
            {"input": "I love this!", "output": "POSITIVE"},
            {"input": "Terrible service.", "output": "NEGATIVE"},
            {"input": "Package arrived Tuesday.", "output": "NEUTRAL"},
        ]
        prompt = build_few_shot_prompt(task, examples, "It was fine.")
        assert task in prompt, 'task_description not found in the returned prompt'
        print(f'{_PASS} Check 3/{total}: task_description appears in the prompt')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 3/{total}: {e}')

    # Check 4: all example inputs, outputs, and new_input appear in the prompt
    try:
        task = "Fix the grammar."
        examples = [
            {"input": "she go to store", "output": "She goes to the store."},
            {"input": "they was happy", "output": "They were happy."},
        ]
        new_input = "he don't know nothing"
        prompt = build_few_shot_prompt(task, examples, new_input)
        for ex in examples:
            assert ex["input"] in prompt, f'example input {ex["input"]!r} not found in prompt'
            assert ex["output"] in prompt, f'example output {ex["output"]!r} not found in prompt'
        assert new_input in prompt, f'new_input {new_input!r} not found in prompt'
        print(f'{_PASS} Check 4/{total}: all example inputs/outputs and new_input appear in the prompt')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 4/{total}: {e}')

    # Check 5: empty examples list runs without error (zero-shot baseline)
    # This makes the zero-shot vs few-shot difference visible: run this prompt
    # through the model yourself and compare the output consistency to Check 3.
    try:
        zero_shot_prompt = build_few_shot_prompt("Classify the sentiment.", [], "It was okay.")
        assert isinstance(zero_shot_prompt, str), 'expected str with empty examples'
        assert len(zero_shot_prompt.strip()) > 0, 'returned empty string with no examples'
        print(f'{_PASS} Check 5/{total}: empty examples list runs without error (zero-shot baseline)')
        print('         ↳ Compare this prompt to the few-shot version — the model\'s output format')
        print('           is less consistent without examples. That is the few-shot advantage.')
        score += 1
    except Exception as e:
        print(f'{_FAIL} Check 5/{total}: {e}')

    print()
    if score == total:
        print('🎉 Exercise complete!')
        print(f'  {total}/{total} passed.')
    else:
        print(f'  {score}/{total} passed. Keep going!')

_run_checks()

## Bonus Challenge

On Day 10 you will build parameterised prompt templates. As a preview, refactor `build_few_shot_prompt` to accept an optional `output_label: str = "Output"` parameter so callers can rename the output line (e.g. `"Label"` for classification or `"Translation"` for a translation task). Update the docstring and verify your checks still pass.

```python
# Example target behaviour:
prompt = build_few_shot_prompt(
    "Translate English to French.",
    [{"input": "Hello", "output": "Bonjour"}],
    "Goodbye",
    output_label="Translation",
)
# The prompt should say "Translation: Bonjour" instead of "Output: Bonjour"
```

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def build_few_shot_prompt(task_description: str, examples: list[dict], new_input: str) -> str:
    """Build a few-shot prompt string from a task description, examples, and a new input."""
    lines = [f"Task: {task_description}", ""]
    for i, ex in enumerate(examples, start=1):
        lines.append(f"Example {i}:")
        lines.append(f"Input: {ex['input']}")
        lines.append(f"Output: {ex['output']}")
        lines.append("")
    lines.append("Now do the same for:")
    lines.append(f"Input: {new_input}")
    lines.append("Output:")
    return "\n".join(lines)
```

**Why this works:** the function enforces a fixed, readable structure — task header, numbered example blocks, then the live input — so the model always sees the same layout regardless of which task or examples you pass in. Ending with `Output:` is a classic completion cue: the model continues the pattern by writing the answer on the very next line, which is exactly the behaviour you want from a few-shot prompt.
</details>